# Employee Onboarding Automation System

## 📚 Learning Objectives
This notebook demonstrates:
- **Custom Tool Creation**: Building specialized tools for specific business logic
- **Asynchronous Execution**: Running multiple crew workflows in parallel
- **Input Templating**: Using placeholders to process multiple similar requests
- **Sequential Workflows**: Coordinating tasks that depend on previous results

## 🎯 Business Case
Automate the employee onboarding process by:
1. Validating start dates (avoiding holidays/weekends)
2. Generating company email addresses
3. Providing PC setup instructions
4. Creating comprehensive onboarding reports

In [ ]:
# ============================================
# IMPORTS AND SETUP
# ============================================
import asyncio  # For asynchronous execution of multiple workflows
from crewai import Agent, Crew, Task, Process  # Core CrewAI components
from crewai.tools import BaseTool  # Base class for custom tool creation
from pydantic import Field, BaseModel  # For tool input validation
from typing import Type  # For type hints
import re  # For string manipulation in email generation
import os

# Set up API key - Replace with your actual OpenAI API key
os.environ['OPENAI_API_KEY'] = "YOUR_OPENAI_API_KEY"

### Custom Tool Development

**Why Create Custom Tools?**
- Encapsulate business logic (e.g., email format rules)
- Ensure consistent behavior across all agent interactions
- Enable reusability across different crews
- Provide structured input/output validation

In [ ]:
# ============================================
# CUSTOM TOOL: EMAIL GENERATOR
# ============================================

# STEP 1: Define input schema using Pydantic
# This ensures the tool receives properly formatted data
class EmailGeneratorInput(BaseModel):
    """Defines the required inputs for email generation"""
    fullname: str = Field(..., description="The full name of the employee.")
    department: str = Field(..., description="The department of the employee.")

# STEP 2: Create the custom tool by inheriting from BaseTool
class EmailGeneratorTool(BaseTool):
    """
    Custom tool that generates standardized company email addresses.
    
    Format: firstname.lastname@department.company.com
    
    Features:
    - Removes special characters
    - Converts to lowercase
    - Follows company naming convention
    """
    
    # Tool metadata (required)
    name: str = "Email Generator"
    description: str = "Generates a company email based on the employee's full name and department."
    args_schema: Type[BaseModel] = EmailGeneratorInput  # Links to input schema

    def _run(self, fullname: str, department: str) -> str:
        """
        Core tool logic - executed when the agent uses this tool.
        
        Args:
            fullname: Employee's full name (e.g., "John Doe")
            department: Employee's department (e.g., "Engineering")
            
        Returns:
            Generated email address (e.g., "john.doe@engineering.company.com")
        """
        # Clean and format the name (replace spaces with dots, remove special chars)
        name_part = re.sub(r'[^a-zA-Z]', '', fullname.replace(' ', '.')).lower()
        
        # Clean and format department
        department_part = department.replace(' ', '').lower()
        
        # Return formatted email
        return f"{name_part}@{department_part}.company.com"

### Agent Definitions

**Agent Specialization Strategy:**
- Each agent has a **specific role** in the onboarding pipeline
- Agents work **sequentially** - each depends on the previous agent's output
- This mirrors real-world HR processes

In [ ]:
# ============================================
# AGENT DEFINITIONS
# ============================================
# Create three specialized agents for the onboarding workflow

# AGENT 1: Date Validation
scheduler_agent = Agent(
    role="Scheduler",
    goal="Validate and confirm employee start date",
    backstory="Ensures the selected start date is valid and not a holiday.",
    # This agent checks if dates are workdays and suggests alternatives if needed
)

# AGENT 2: IT Setup (with Custom Tool)
it_agent = Agent(
    role="IT Administrator",
    goal="Set up company email and provide PC setup instructions",
    backstory="Prepares company email accounts and delivers simple PC setup guidelines.",
    
    # KEY FEATURE: This agent has access to the custom EmailGeneratorTool
    # The agent will automatically use this tool when it needs to create emails
    tools=[EmailGeneratorTool()]
)

# AGENT 3: Report Compilation
report_agent = Agent(
    role="Onboarding Reporter",
    goal="Generate an onboarding summary report",
    backstory="Creates a final onboarding report with all key details for HR records.",
    # This agent consolidates information from previous agents
)

### Task Configuration

**Input Templating with {placeholders}:**
- Tasks use `{employe}` placeholder to accept dynamic input
- At runtime, CrewAI replaces placeholders with actual values from `inputs` dictionary
- This enables processing multiple employees with the same task definitions

**Note:** There's a typo in the placeholder - it should be `{employee}` instead of `{employe}`. The code works because it consistently uses the same key in the inputs dictionary.

In [ ]:
# ============================================
# TASK DEFINITIONS WITH INPUT TEMPLATING
# ============================================

# TASK 1: Validate Start Date
validate_start_date_task = Task(
    description=(
        "Check if the start date ({employe}) is a working day. "
        "If it's a holiday or weekend, suggest the next available working day."
    ),
    expected_output="Confirmed valid start date or suggestion for a new date.",
    agent=scheduler_agent
    # This task receives employee data through the {employe} placeholder
)

# TASK 2: IT Infrastructure Setup
it_setup_task = Task(
    description=(
        "Create a company email for {employe} in the {employe} department. "
        "Provide a short, clear set of bullet points on how to set up their PC."
    ),
    expected_output="Active company email and clear PC setup instructions.",
    agent=it_agent
    # The IT agent will automatically invoke EmailGeneratorTool when needed
)

# TASK 3: Generate Comprehensive Report
generate_report_task = Task(
    description=(
        "Generate a summary onboarding report for {employe}, starting on {employe}, "
        "in the {employe} department. Include email details, PC setup info, and start date confirmation."
    ),
    expected_output="A clear and concise onboarding summary report.",
    agent=report_agent
    # This task aggregates outputs from Tasks 1 and 2
)


### Crew Assembly

**Process Types:**
- **Sequential**: Tasks execute one after another (used here)
- **Hierarchical**: Manager coordinates tasks
- **Parallel**: Tasks run simultaneously

We use sequential processing because each task depends on the previous one.

In [ ]:
# ============================================
# CREW DEFINITION
# ============================================
# Assemble the onboarding workflow crew

onboarding_crew = Crew(
    # All three agents in execution order
    agents=[scheduler_agent, it_agent, report_agent],
    
    # Tasks will execute in this order (1 → 2 → 3)
    tasks=[validate_start_date_task, it_setup_task, generate_report_task],
    
    # SEQUENTIAL PROCESS: Each task waits for the previous to complete
    # Task 2 can use outputs from Task 1
    # Task 3 can use outputs from Tasks 1 and 2
    process=Process.sequential,
    
    # Show detailed logs for educational purposes
    verbose=True
)

### Asynchronous Execution

**Why Use Async?**
- Process multiple employees **concurrently** (not sequentially)
- Significantly reduces total processing time for batch operations
- Example: 3 employees processed in parallel vs. one-by-one

**Technical Note:**
- `kickoff_async()` returns a coroutine that runs the crew workflow
- `asyncio.run()` executes the async function
- `nest_asyncio.apply()` allows async execution in Jupyter notebooks

In [ ]:
# ============================================
# ASYNCHRONOUS EXECUTION FUNCTION
# ============================================
# This function enables concurrent processing of multiple employees

async def async_onboarding_execution(onboarding_inputs):
    """
    Runs the onboarding crew asynchronously.
    
    Benefits:
    - Can process multiple employees in parallel
    - Non-blocking execution
    - Faster batch processing
    
    Args:
        onboarding_inputs: Dictionary with employee data
    """
    # Execute crew with async method
    results = await onboarding_crew.kickoff_async(inputs=onboarding_inputs)
    
    # Display the final report
    print("Crew result:", results.raw)

### Execution: Processing Multiple Employees

**What Happens Here:**
1. We pass 3 employees' data in a single inputs dictionary
2. The crew processes all employees (in this case, sequentially due to the current setup)
3. Each employee goes through the full pipeline: date validation → IT setup → report

**For True Parallel Processing:**
You could create multiple crew instances and run them with `asyncio.gather()`:
```python
await asyncio.gather(
    crew1.kickoff_async(inputs=employee1),
    crew2.kickoff_async(inputs=employee2),
    crew3.kickoff_async(inputs=employee3)
)
```

In [ ]:
# ============================================
# MAIN EXECUTION
# ============================================

# SETUP: Allow async execution in Jupyter notebooks
import nest_asyncio
nest_asyncio.apply()

# INPUTS: Employee data for onboarding
# The "employe" key matches the placeholder in task descriptions
onboarding_inputs = {
    "employe": [
        "Alice Johnson, starting 2024-07-15, department Engineering",
        "Bob Smith, starting 2024-07-16, department Marketing",
        "Charlie Brown, starting 2024-07-17, department Sales"
    ]
}

# EXECUTION: Run the async onboarding workflow
# This will:
# 1. Validate start dates for all employees (Scheduler Agent)
# 2. Generate emails using EmailGeneratorTool (IT Agent)
# 3. Create PC setup instructions (IT Agent)
# 4. Compile comprehensive onboarding reports (Report Agent)
asyncio.run(async_onboarding_execution(onboarding_inputs))

# EXPECTED OUTPUTS:
# - Validated start dates (all are weekdays - no changes needed)
# - Email addresses: alicejohnson@engineering.company.com, etc.
# - PC setup instructions (10-step guide)
# - Final onboarding report for HR records